# Vehicle Budget Recommender - Model Training

**Goal:** the user enters a budget (e.g. Rs 150,000) and the app tells them what they can buy -
from a cycle to a helicopter - including combinations like *"2 cycles + 1 electric bike"*.
The deployed app also includes **V-Bot**, a floating chatbot that answers questions from the same dataset.

**This notebook does 4 things**
1. Loads and checks the dataset (`vehicles.csv`)
2. Trains a **Random Forest** model that estimates a vehicle's price from its specs
3. Trains a **KNN (K-Nearest Neighbours)** model that finds similar vehicles
4. Saves everything to `model.joblib` (used by the Streamlit app)

**Kaggle setup:** upload `vehicles.csv` as a Kaggle *Dataset*, add it to this notebook with
**Add Input**, then press **Run All**.

## 1. Imports and finding the dataset

In [ ]:
import glob
import itertools
import os
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# On Kaggle the file lives under /kaggle/input/<your-dataset-name>/vehicles.csv
paths = glob.glob("/kaggle/input/**/vehicles.csv", recursive=True) + glob.glob("vehicles.csv")
if not paths:
    raise FileNotFoundError(
        "vehicles.csv not found. Upload it as a Kaggle Dataset and click 'Add Input' on the right."
    )
DATA_PATH = paths[0]
print("Using dataset:", DATA_PATH)

## 2. Load and check the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape (rows, columns):", df.shape)
df.head(10)

In [ ]:
df.info()
print()
print(df["price_pkr"].describe().apply(lambda v: f"{v:,.0f}"))

In [ ]:
# --- basic cleaning / sanity checks ---
df["name"] = df["name"].str.strip()
df["category"] = df["category"].str.strip()
df = df.drop_duplicates(subset="name").dropna(subset=["name", "category", "price_pkr"])
df = df[df["price_pkr"] > 0].reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("Missing values:", int(df.isna().sum().sum()))
print()
print(df["category"].value_counts())

## 3. Look at the data
Prices range from Rs 15,000 (a kids cycle) to Rs 19 billion (a private jet), so the second chart
uses a **log scale** - otherwise every cheap vehicle would sit at zero.

In [ ]:
DOT = "#2B6CB0"
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df["category"].value_counts()
axes[0].barh(counts.index[::-1], counts.values[::-1], color=DOT, height=0.6)
axes[0].set_title("Vehicles per category", loc="left")
axes[0].set_xlabel("Number of vehicles in the dataset")

med = df.groupby("category")["price_pkr"].median().sort_values()
axes[1].scatter(med.values, med.index, color=DOT, s=60)
axes[1].set_xscale("log")
axes[1].set_title("Median price per category", loc="left")
axes[1].set_xlabel("Price in PKR (log scale)")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color="#E2E8F0", linewidth=0.8)
    ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 4. Helper functions
`add_features`, `find_combos`, `similar_vehicles`, `predict_price` and `money`.
These are the same functions the Streamlit app uses (`recommender.py`), so the notebook and the app behave identically.

* `find_combos` is a small search (like the *knapsack problem*): it tries combinations of up to 3 vehicles,
  keeps those that fit the budget and use at least 85% of it, and ranks them by budget used + variety.

In [ ]:
PRICE_FEATURES = ["category", "fuel_type", "top_speed_kmh", "seats"]
SIMILARITY_FEATURES = ["category", "fuel_type", "log_price", "top_speed_kmh", "seats"]


def add_features(df):
    """Adds log_price. Prices range from Rs 15,000 to Rs 19 billion, so we use log10."""
    data = df.copy().reset_index(drop=True)
    data["log_price"] = np.log10(data["price_pkr"])
    return data


def money(x):
    """Rs 150,000 (1.50 lakh) / Rs 3,000,000 (30.00 lakh) / Rs 90,000,000 (9.00 crore) ..."""
    x = float(x)
    if x >= 1e9:
        words = f"{x / 1e9:.2f} arab"
    elif x >= 1e7:
        words = f"{x / 1e7:.2f} crore"
    elif x >= 1e5:
        words = f"{x / 1e5:.2f} lakh"
    else:
        words = None
    text = f"Rs {x:,.0f}"
    return f"{text} ({words})" if words else text


def find_combos(df, budget, max_items=3, top_n=10, min_spend_ratio=0.85, pool_size=40):
    """
    Finds combinations of vehicles whose total price fits inside the budget.

    - The same vehicle can be repeated (e.g. 2 cycles).
    - A combination must use at least `min_spend_ratio` of the budget (default 85%),
      otherwise "1 cheap cycle" would always be a valid answer.
    - Combinations are ranked by: how much of the budget is used + how many
      different vehicle categories they contain (variety).
    - At most 2 combinations with the same category mix are shown.
    """
    pool = df[df["price_pkr"] <= budget].sort_values("price_pkr").reset_index(drop=True)
    if pool.empty:
        return []

    # keep the search fast for big budgets: spread the pool evenly across the price range
    if len(pool) > pool_size:
        keep = np.unique(np.linspace(0, len(pool) - 1, pool_size).round().astype(int))
        pool = pool.iloc[keep].reset_index(drop=True)

    names = pool["name"].tolist()
    prices = pool["price_pkr"].to_numpy()
    cats = pool["category"].tolist()

    found = []
    for size in range(1, max_items + 1):
        for combo in itertools.combinations_with_replacement(range(len(pool)), size):
            total = float(sum(prices[i] for i in combo))
            if total > budget or total < min_spend_ratio * budget:
                continue
            variety = len({cats[i] for i in combo})
            score = total / budget + 0.10 * variety
            found.append((score, combo, total))

    found.sort(key=lambda t: t[0], reverse=True)

    results, mix_count = [], Counter()
    for score, combo, total in found:
        mix = tuple(sorted(cats[i] for i in combo))
        if mix_count[mix] >= 2:
            continue
        mix_count[mix] += 1
        counts = Counter(combo)
        items = [
            {"name": names[i], "category": cats[i], "qty": q, "unit_price": float(prices[i])}
            for i, q in counts.items()
        ]
        summary = " + ".join(f"{it['qty']} x {it['name']}" for it in items)
        results.append(
            {"summary": summary, "items": items, "total": total, "remaining": float(budget - total)}
        )
        if len(results) >= top_n:
            break
    return results


def similar_vehicles(df, bundle, name, max_price=None, k=5):
    """KNN: the k vehicles most similar to `name` (category, fuel, price, speed, seats)."""
    data = add_features(df)
    X = bundle["feat"].transform(data[SIMILARITY_FEATURES])
    pos = int(np.where(data["name"] == name)[0][0])
    _, idx = bundle["knn"].kneighbors(X[[pos]], n_neighbors=len(data))
    rows = [i for i in idx[0] if i != pos]
    out = data.iloc[rows]
    if max_price is not None:
        out = out[out["price_pkr"] <= max_price]
    return out.head(k)[["name", "category", "price_pkr", "fuel_type", "seats", "top_speed_kmh"]]


def predict_price(bundle, category, fuel_type, seats, top_speed_kmh):
    """Random Forest price estimate in PKR."""
    row = pd.DataFrame(
        [{"category": category, "fuel_type": fuel_type, "top_speed_kmh": top_speed_kmh, "seats": seats}]
    )
    return float(10 ** bundle["price_model"].predict(row[PRICE_FEATURES])[0])

## 5. Model 1 - Random Forest: estimate a vehicle's price
We predict **log10(price)** from `category`, `fuel_type`, `top_speed_kmh` and `seats`.
(Log, because prices span 6 orders of magnitude.)
We first test the model on data it has not seen (80/20 split, plus 5-fold cross-validation),
then train the final model on all the rows.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_price_model():
    pre = ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["category", "fuel_type"]),
            ("num", "passthrough", ["top_speed_kmh", "seats"]),
        ]
    )
    return Pipeline([("pre", pre), ("rf", RandomForestRegressor(n_estimators=300, random_state=42))])


data = add_features(df)
X, y = data[PRICE_FEATURES], data["log_price"]

# 1) hold-out test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
test_model = build_price_model().fit(X_train, y_train)
pred = test_model.predict(X_test)
print(f"Hold-out R2 (log scale): {r2_score(y_test, pred):.3f}")
print(f"Hold-out MAE (log10)   : {mean_absolute_error(y_test, pred):.3f}  "
      f"-> typical estimate is within a factor of {10 ** mean_absolute_error(y_test, pred):.2f} of the real price")

# 2) cross-validation (more reliable on a small dataset)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(build_price_model(), X, y, cv=cv, scoring="r2")
print(f"5-fold CV R2           : {scores.mean():.3f} (+/- {scores.std():.3f})")

# 3) final model on ALL rows
price_model = build_price_model().fit(X, y)

## 6. Model 2 - KNN: find similar vehicles
Every vehicle becomes a point described by category, fuel type, (log) price, top speed and seats.
KNN finds the points closest to a chosen vehicle - used in the app as *"similar options"*.

In [ ]:
feat = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["category", "fuel_type"]),
        ("num", StandardScaler(), ["log_price", "top_speed_kmh", "seats"]),
    ]
)
X_sim = feat.fit_transform(data[SIMILARITY_FEATURES])
knn = NearestNeighbors(metric="euclidean").fit(X_sim)

bundle = {"price_model": price_model, "feat": feat, "knn": knn, "n_rows": len(data)}
print("KNN trained on", X_sim.shape[0], "vehicles with", X_sim.shape[1], "features")

## 7. Try it out

In [ ]:
BUDGET = 150_000
print(f"Combinations for a budget of {money(BUDGET)}:\n")
for combo in find_combos(df, BUDGET, max_items=3, top_n=5):
    print(" -", combo["summary"])
    print("     total", money(combo["total"]), "| left", money(combo["remaining"]))

In [ ]:
print("Vehicles similar to Honda CD 70 (under Rs 150,000):")
similar_vehicles(df, bundle, "Honda CD 70", max_price=150_000, k=5)

In [ ]:
est = predict_price(bundle, category="Petrol Bike", fuel_type="Petrol", seats=2, top_speed_kmh=100)
print("Estimated price of a 100 km/h petrol bike:", money(est))

## 8. Save the model
After the notebook finishes, open the **Output** tab on the right and download `model.joblib`.
Upload it to GitHub next to `app.py`.

In [ ]:
OUT = "/kaggle/working/model.joblib" if os.path.isdir("/kaggle/working") else "model.joblib"
joblib.dump(bundle, OUT)
print("Saved:", OUT, f"({os.path.getsize(OUT) / 1024:.0f} KB)")

# check it loads again and still works
check = joblib.load(OUT)
print("Reloaded OK. Estimate for an electric 60 km/h bike:",
      money(predict_price(check, "Electric Bike", "Electric", 2, 60)))